In [1]:
pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 2.1 MB/s eta 0:00:00


In [2]:
import qiskit
print(qiskit.__version__)

2.5.2


Import Required Libraries

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import qiskit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error

Exercise 1 Easy - Implement Superdense Coding for Message '00' & Verify Decoding

In [4]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
# Construct superdense coding pipeline for payload '00'
sdc_circ_00 = QuantumCircuit(2, 2)
# Step 1: EPR generation
sdc_circ_00.h(0)
sdc_circ_00.cx(0, 1)
sdc_circ_00.barrier()
# Step 2: Alice encodes '00' (Identity / pass-through)
sdc_circ_00.id(0)
sdc_circ_00.barrier()
# Step 3: Bob performs Bell measurement
sdc_circ_00.cx(0, 1)
sdc_circ_00.h(0)
sdc_circ_00.barrier()
sdc_circ_00.measure([0, 1], [0, 1])
backend_sim = AerSimulator()
counts_out = backend_sim.run(sdc_circ_00, shots=1024).result().get_counts()
print("--- Superdense Coding Circuit (Message: '00') ---")
print(sdc_circ_00)
print("\nMeasured Outcome Frequencies:", counts_out)
status_str = "SUCCESS" if list(counts_out.keys())[0] == '00' else "FAIL"
print(f"Decoded Outcome: {list(counts_out.keys())[0]} | Protocol Status: {status_str}")

--- Superdense Coding Circuit (Message: '00') ---
     ┌───┐      ░ ┌───┐ ░      ┌───┐ ░ ┌─┐   
q_0: ┤ H ├──■───░─┤ I ├─░───■──┤ H ├─░─┤M├───
     └───┘┌─┴─┐ ░ └───┘ ░ ┌─┴─┐└───┘ ░ └╥┘┌─┐
q_1: ─────┤ X ├─░───────░─┤ X ├──────░──╫─┤M├
          └───┘ ░       ░ └───┘      ░  ║ └╥┘
c: 2/═══════════════════════════════════╩══╩═
                                        0  1 

Measured Outcome Frequencies: {'00': 1024}
Decoded Outcome: 00 | Protocol Status: SUCCESS


Exercise 2 Medium -
Implement Superdense Coding for All Four 2-Bit Messages & Tabulate Results

In [5]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
def build_dense_coding_circuit(bitstring):
    qc = QuantumCircuit(2, 2)
    # Entanglement generation
    qc.h(0)
    qc.cx(0, 1)
    qc.barrier()
    # Alice's local encoding
    if bitstring[1] == '1':
        qc.x(0)
    if bitstring[0] == '1':
        qc.z(0)
    qc.barrier()
    # Bob's decoding
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])
    return qc
bitstrings = ['00', '01', '10', '11']
sim_engine = AerSimulator()
print("--- Multi-Message Superdense Coding Simulation (1024 Shots) ---")
print(f"{'Input Bits':<15}{'Encoding Operator':<20}{'Resulting Bell State':<22}{'Output':<15}{'Check'}")
print("-" * 80)
gate_map = {'00': 'I', '01': 'X', '10': 'Z', '11': 'X followed by Z'}
bell_map = {'00': '|Phi+>', '01': '|Psi+>', '10': '|Phi->', '11': '|Psi->'}
for b in bitstrings:
    c = build_dense_coding_circuit(b)
    res = sim_engine.run(c, shots=1024).result().get_counts()
    extracted = list(res.keys())[0]
    verdict = "PASSED" if extracted == b else "FAILED"
    print(f"{b:<15}{gate_map[b]:<20}{bell_map[b]:<22}{str(res):<15}{verdict}")

--- Multi-Message Superdense Coding Simulation (1024 Shots) ---
Input Bits     Encoding Operator   Resulting Bell State  Output         Check
--------------------------------------------------------------------------------
00             I                   |Phi+>                {'00': 1024}   PASSED
01             X                   |Psi+>                {'10': 1024}   FAILED
10             Z                   |Phi->                {'01': 1024}   FAILED
11             X followed by Z     |Psi->                {'11': 1024}   PASSED


Exercise 3 Hard -
Channel Noise Simulation: Bit-Flip Error on Transmitted Qubit

In [6]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
def simulate_noisy_transit(bitstring):
    qc = QuantumCircuit(2, 2)
    qc.h(0)
    qc.cx(0, 1)
    # Alice encodes
    if bitstring[1] == '1': qc.x(0)
    if bitstring[0] == '1': qc.z(0)
    # Noise Event: Bit-flip during transmission of Alice's qubit
    qc.barrier()
    qc.x(0)
    qc.barrier()
    # Bob decodes
    qc.cx(0, 1)
    qc.h(0)
    qc.measure([0, 1], [0, 1])
    return qc
print("--- Analysis of Transmission Channel Bit-Flip Error ---")
print(f"{'Sent Payload':<16}{'Channel Noise':<20}{'Received Payload':<20}{'Error Description'}")
print("-" * 75)
for b in bitstrings:
    circ_noisy = simulate_noisy_transit(b)
    out = sim_engine.run(circ_noisy, shots=1000).result().get_counts()
    recv_str = list(out.keys())[0]
    print(f"{b:<16}{'Bit-Flip X on qA':<20}{recv_str:<20}{f'Payload {b} mapped to {recv_str}'}")

--- Analysis of Transmission Channel Bit-Flip Error ---
Sent Payload    Channel Noise       Received Payload    Error Description
---------------------------------------------------------------------------
00              Bit-Flip X on qA    10                  Payload 00 mapped to 10
01              Bit-Flip X on qA    00                  Payload 01 mapped to 00
10              Bit-Flip X on qA    11                  Payload 10 mapped to 11
11              Bit-Flip X on qA    01                  Payload 11 mapped to 01


Exercise 4 Real-world -
Classical Bandwidth vs. Superdense Coding for a 100-Message Session

In [7]:
# Transmission efficiency modeling for 100 2-bit messages
n_messages = 100
bits_per_packet = 2
total_data_bits = n_messages * bits_per_packet
conventional_bits_sent = total_data_bits
quantum_carriers_sent = n_messages
efficiency_gain = conventional_bits_sent / quantum_carriers_sent
print("--- Communication Session Resource Analysis ---")
print(f"Total Classical Payload:               {total_data_bits} bits")
print(f"Conventional Channel Requirement:      {conventional_bits_sent} classical transmissions")
print(f"Superdense Coding Channel Requirement: {quantum_carriers_sent} qubit transmissions")
print(f"Channel Utilization Gain Factor:       {efficiency_gain:.1f}x")
print(f"Carrier Savings:                       50% reduction in active channel transmissions")

--- Communication Session Resource Analysis ---
Total Classical Payload:               200 bits
Conventional Channel Requirement:      200 classical transmissions
Superdense Coding Channel Requirement: 100 qubit transmissions
Channel Utilization Gain Factor:       2.0x
Carrier Savings:                       50% reduction in active channel transmissions


Exercise 5 Challenge -
Protocol Execution with Qiskit Noise Model & Decoding Error Rate Estimation

In [11]:
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
# Configure depolarizing channel noise model
depol_rate = 0.06  # 6% depolarizing noise on transit qubit
custom_noise = NoiseModel()
error_gate = depolarizing_error(depol_rate, 1)
custom_noise.add_quantum_error(error_gate, ['id', 'x', 'z'], [0])
noisy_backend = AerSimulator(noise_model=custom_noise)
num_shots = 5000
print("--- Dense Coding Error Estimation Under Depolarizing Noise (6%) ---")
print(f"{'Message':<14}{'Correct Decodes':<18}{'Corrupted Decodes':<20}{'Error Percentage'}")
print("-" * 68)
for b in bitstrings:
    c_noisy = build_dense_coding_circuit(b)
    counts_noisy = noisy_backend.run(c_noisy, shots=num_shots).result().get_counts()
    correct_count = counts_noisy.get(b, 0)
    failed_count = num_shots - correct_count
    err_pct = (failed_count / num_shots) * 100
    print(f"{b:<14}{correct_count:<18}{failed_count:<20}{err_pct:.2f}%")


--- Dense Coding Error Estimation Under Depolarizing Noise (6%) ---
Message       Correct Decodes   Corrupted Decodes   Error Percentage
--------------------------------------------------------------------
00            5000              0                   0.00%
01            76                4924                98.48%
10            84                4916                98.32%
11            4540              460                 9.20%
